In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/processed/merged_data.csv")

In [4]:
print("Original shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Original shape: (456548, 15)

Columns:
['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders', 'city_code', 'region_code', 'center_type', 'op_area', 'category', 'cuisine']


In [5]:
print("Unique centers:", df['center_id'].nunique())
print("Unique meals:", df['meal_id'].nunique())
print("Unique weeks:", df['week'].nunique())

print("\nWeek range:")
print(df['week'].min(), "to", df['week'].max())

Unique centers: 77
Unique meals: 51
Unique weeks: 145

Week range:
1 to 145


Reindex the panel to a full (center_id, meal_id, week) grid.

Why: the raw data only has a row when a meal was actually ordered.
A missing row does NOT mean "unknown" — it means the meal had 0 orders that week. If we don't fill these gaps, later lag/rolling features (e.g. "orders last week") will silently pull from the wrong week whenever there's a gap, corrupting the feature without any error.

In [6]:
min_week, max_week = df['week'].min(), df['week'].max()
full_weeks = pd.RangeIndex(min_week, max_week + 1, name='week')

print(f"Full week range: {min_week} to {max_week} ({len(full_weeks)} weeks)")

Full week range: 1 to 145 (145 weeks)


Only reindex pairs that actually existed at least once in the data. We do NOT want to invent center-meal combinations that never happened (e.g. a meal a center never sells).

In [7]:
real_pairs = df[['center_id', 'meal_id']].drop_duplicates()
len(real_pairs)

3597

In [8]:
# Cross join: every real (center_id, meal_id) pair x every week in range
full_grid = real_pairs.merge(pd.DataFrame({'week': full_weeks}), how='cross')
print(f"Full grid shape: {full_grid.shape}")
full_grid.sample(10)

Full grid shape: (521565, 3)


,center_id,meal_id,week
449519,57,2704,20
73018,89,2290,84
170560,177,2640,41
99880,108,1207,121
173681,27,2139,117
157217,36,1847,38
313675,30,1847,41
243773,50,2290,29
36468,109,2290,74
261633,113,2640,54


In [9]:
# Left-merge real data onto the full grid. Weeks that didn't exist originally come back as NaN across every non-key column.
df_full = full_grid.merge(df, on=['center_id', 'meal_id', 'week'], how='left')
nan_counts = df_full.isna().sum()
print(nan_counts)

center_id                    0
meal_id                      0
week                         0
id                       65017
checkout_price           65017
base_price               65017
emailer_for_promotion    65017
homepage_featured        65017
num_orders               65017
city_code                65017
region_code              65017
center_type              65017
op_area                  65017
category                 65017
cuisine                  65017
dtype: int64


In [10]:
# num_orders: NaN here genuinely means zero orders that week -> fill 0
df_full['num_orders'] = df_full['num_orders'].fillna(0)
nan_counts = df_full.isna().sum()
print(nan_counts)

center_id                    0
meal_id                      0
week                         0
id                       65017
checkout_price           65017
base_price               65017
emailer_for_promotion    65017
homepage_featured        65017
num_orders                   0
city_code                65017
region_code              65017
center_type              65017
op_area                  65017
category                 65017
cuisine                  65017
dtype: int64


In [11]:
# id was only there to identify original rows, not needed as a feature
df_full.drop(columns=['id'], errors='ignore', inplace=True)
df_full.sample(10)

,center_id,meal_id,week,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,center_type,op_area,category,cuisine
481102,101,2569,138,286.18,285.18,0.0,0.0,337.0,699.0,85.0,TYPE_C,2.8,Salad,Italian
356539,80,1727,130,466.63,464.63,0.0,0.0,674.0,604.0,56.0,TYPE_C,5.1,Rice Bowl,Indian
34353,13,2492,134,290.09,389.03,0.0,0.0,352.0,590.0,56.0,TYPE_B,6.7,Desert,Indian
440120,26,1445,46,669.33,670.33,0.0,0.0,26.0,515.0,77.0,TYPE_C,3.0,Seafood,Continental
229576,81,2640,42,280.33,281.33,0.0,0.0,109.0,526.0,34.0,TYPE_A,4.0,Starters,Thai
319810,76,2581,86,486.03,679.03,0.0,0.0,230.0,614.0,85.0,TYPE_A,3.0,Pizza,Continental
8973,24,1311,129,186.30,226.01,0.0,0.0,298.0,614.0,85.0,TYPE_B,3.6,Extras,Thai
359078,80,2494,59,242.53,243.53,0.0,0.0,67.0,604.0,56.0,TYPE_C,5.1,Soup,Thai
251275,104,1902,136,447.23,446.23,0.0,0.0,14.0,647.0,56.0,TYPE_A,4.5,Biryani,Indian
193102,64,2126,108,532.56,532.56,0.0,0.0,26.0,553.0,77.0,TYPE_A,4.4,Pasta,Italian


In [12]:
# Sort so groupby + ffill/bfill operate in correct week order
df_full = df_full.sort_values(['center_id', 'meal_id', 'week']).reset_index(drop=True)

In [13]:
# Static attributes (never change for a given center_id or meal_id):
# just carry the known value across the gap weeks for that pair.
static_cols = ['city_code', 'region_code', 'center_type', 'op_area',
               'category', 'cuisine']

for col in static_cols:
    df_full[col] = df_full.groupby(['center_id', 'meal_id'])[col].transform(
        lambda s: s.ffill().bfill()
    )

nan_counts = df_full.isna().sum()
print(nan_counts)

center_id                    0
meal_id                      0
week                         0
checkout_price           65017
base_price               65017
emailer_for_promotion    65017
homepage_featured        65017
num_orders                   0
city_code                    0
region_code                  0
center_type                  0
op_area                      0
category                     0
cuisine                      0
dtype: int64


Price columns CAN vary week to week, so we assume the price on a gap week was roughly the same as the nearest known week for that same center-meal pair. This is an ASSUMPTION, not a fact. It is worth stating explicitly in the README as a modeling choice.

In [14]:
price_cols = ['checkout_price', 'base_price']

for col in price_cols:
    df_full[col] = df_full.groupby(['center_id', 'meal_id'])[col].transform(
        lambda s: s.ffill().bfill()
    )

nan_counts = df_full.isna().sum()
print(nan_counts)

center_id                    0
meal_id                      0
week                         0
checkout_price               0
base_price                   0
emailer_for_promotion    65017
homepage_featured        65017
num_orders                   0
city_code                    0
region_code                  0
center_type                  0
op_area                      0
category                     0
cuisine                      0
dtype: int64


Promo flags: if there was no order row, no promotion was running -> 0

In [15]:
df_full['emailer_for_promotion'] = df_full['emailer_for_promotion'].fillna(0).astype(int)
df_full['homepage_featured'] = df_full['homepage_featured'].fillna(0).astype(int)

nan_counts = df_full.isna().sum()
print(nan_counts)

center_id                0
meal_id                  0
week                     0
checkout_price           0
base_price               0
emailer_for_promotion    0
homepage_featured        0
num_orders               0
city_code                0
region_code              0
center_type              0
op_area                  0
category                 0
cuisine                  0
dtype: int64


In [16]:
# Final check: confirm no NaNs remain anywhere
print("\nRemaining NaNs per column:")
print(df_full.isnull().sum())

print(f"\nFinal shape: {df_full.shape}")


Remaining NaNs per column:
center_id                0
meal_id                  0
week                     0
checkout_price           0
base_price               0
emailer_for_promotion    0
homepage_featured        0
num_orders               0
city_code                0
region_code              0
center_type              0
op_area                  0
category                 0
cuisine                  0
dtype: int64

Final shape: (521565, 14)


Target transform: compress the long right tail so the model doesn't overweight a handful of huge center-meal combos. +1 handles the genuine zeros we just created via reindexing (log(0) is undefined).

In [17]:
# log1p(x) == log(1 + x)
df_full['log_num_orders'] = np.log1p(df_full['num_orders'])
df_full.sample(10)

,center_id,meal_id,week,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,center_type,op_area,category,cuisine,log_num_orders
28758,14,2760,49,242.53,243.53,0,0,67.0,654.0,56.0,TYPE_C,2.7,Other Snacks,Thai,4.219508
402952,124,1543,143,486.03,484.03,0,0,28.0,590.0,56.0,TYPE_C,4.0,Desert,Indian,3.367296
197326,64,1993,127,116.43,116.43,0,0,310.0,553.0,77.0,TYPE_A,4.4,Beverages,Thai,5.739793
324748,93,2444,94,437.53,708.13,1,0,296.0,461.0,34.0,TYPE_A,3.9,Seafood,Continental,5.693732
400953,113,2867,29,554.87,658.63,0,0,0.0,680.0,77.0,TYPE_C,4.0,Seafood,Continental,0.000000
509141,177,1207,47,317.19,319.19,0,0,203.0,683.0,56.0,TYPE_A,3.4,Beverages,Continental,5.318120
112637,41,1207,118,386.09,387.09,0,0,175.0,590.0,56.0,TYPE_C,1.9,Beverages,Continental,5.170484
180059,58,2760,115,243.53,241.53,0,0,0.0,695.0,77.0,TYPE_C,3.8,Other Snacks,Thai,0.000000
390357,110,1778,18,158.14,181.39,1,0,431.0,485.0,77.0,TYPE_A,3.8,Beverages,Italian,6.068426
72306,29,1247,97,446.23,446.23,0,0,0.0,526.0,34.0,TYPE_C,4.0,Biryani,Indian,0.000000


Lag and rolling features:- all built on the SHIFTED series so no row ever sees its own current-week value. Group by (center_id, meal_id) so features never leak across different centers or meals.

In [18]:
df_full = df_full.sort_values(['center_id', 'meal_id', 'week']).reset_index(drop=True)

In [19]:
grp = df_full.groupby(['center_id', 'meal_id'])['log_num_orders']

In [20]:
# --- Lag features: orders N weeks ago ---
df_full['lag_1'] = grp.shift(1)
df_full['lag_2'] = grp.shift(2)
df_full['lag_4'] = grp.shift(4)

df_full.head(10)

,center_id,meal_id,week,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,center_type,op_area,category,cuisine,log_num_orders,lag_1,lag_2,lag_4
0,10,1062,1,181.39,181.39,0,0,865.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,6.763885,NaN,NaN,NaN
1,10,1062,2,183.36,182.36,0,0,782.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,6.663133,6.763885,NaN,NaN
2,10,1062,3,184.36,182.36,0,0,851.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,6.747587,6.663133,6.763885,NaN
3,10,1062,4,182.36,183.36,0,0,1202.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,7.092574,6.747587,6.663133,NaN
4,10,1062,5,183.39,181.39,0,0,958.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,6.865891,7.092574,6.747587,6.763885
5,10,1062,6,162.05,183.39,0,0,1094.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,6.998510,6.865891,7.092574,6.663133
6,10,1062,7,160.08,183.39,0,0,1513.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,7.322510,6.998510,6.865891,6.747587
7,10,1062,8,160.05,182.39,0,0,1149.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,7.047517,7.322510,6.998510,7.092574
8,10,1062,9,162.05,182.39,0,0,1282.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,7.156956,7.047517,7.322510,6.865891
9,10,1062,10,161.05,181.39,0,0,1473.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,7.295735,7.156956,7.047517,6.998510


--- Rolling features: always shift(1) FIRST, then roll ---

In [21]:
shifted = grp.shift(1)
df_full['rolling_mean_4'] = shifted.groupby([df_full['center_id'], df_full['meal_id']]).transform(
    lambda s: s.rolling(window=4, min_periods=1).mean()
)
df_full['rolling_std_4'] = shifted.groupby([df_full['center_id'], df_full['meal_id']]).transform(
    lambda s: s.rolling(window=4, min_periods=1).std()
)

df_full.head(10)

,center_id,meal_id,week,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,center_type,op_area,category,cuisine,log_num_orders,lag_1,lag_2,lag_4,rolling_mean_4,rolling_std_4
0,10,1062,1,181.39,181.39,0,0,865.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,6.763885,NaN,NaN,NaN,NaN,NaN
1,10,1062,2,183.36,182.36,0,0,782.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,6.663133,6.763885,NaN,NaN,6.763885,NaN
2,10,1062,3,184.36,182.36,0,0,851.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,6.747587,6.663133,6.763885,NaN,6.713509,0.071243
3,10,1062,4,182.36,183.36,0,0,1202.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,7.092574,6.747587,6.663133,NaN,6.724868,0.054082
4,10,1062,5,183.39,181.39,0,0,958.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,6.865891,7.092574,6.747587,6.763885,6.816794,0.189081
5,10,1062,6,162.05,183.39,0,0,1094.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,6.998510,6.865891,7.092574,6.663133,6.842296,0.186427
6,10,1062,7,160.08,183.39,0,0,1513.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,7.322510,6.998510,6.865891,6.747587,6.926140,0.151051
7,10,1062,8,160.05,182.39,0,0,1149.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,7.047517,7.322510,6.998510,7.092574,7.069871,0.192391
8,10,1062,9,162.05,182.39,0,0,1282.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,7.156956,7.047517,7.322510,6.865891,7.058607,0.191937
9,10,1062,10,161.05,181.39,0,0,1473.0,590.0,56.0,TYPE_B,6.3,Beverages,Italian,7.295735,7.156956,7.047517,6.998510,7.131373,0.143611


--- Price change feature (sign convention: positive = markup, negative = discount) ---

In [22]:
df_full['price_change_pct'] = (
    (df_full['checkout_price'] - df_full['base_price']) / df_full['base_price']
) * 100

df_full.sample(10)

,center_id,meal_id,week,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,...,op_area,category,cuisine,log_num_orders,lag_1,lag_2,lag_4,rolling_mean_4,rolling_std_4,price_change_pct
191704,61,2322,15,325.01,324.01,0,0,68.0,473.0,77.0,...,4.5,Beverages,Continental,4.234107,4.007333,4.025352,0.000000,2.936564,1.962919,0.308632
443547,143,1109,138,315.28,314.28,0,0,447.0,562.0,77.0,...,3.8,Rice Bowl,Indian,6.104793,5.828946,5.700444,6.104793,5.754669,0.298928,0.318188
92977,34,1525,33,292.03,292.03,0,0,94.0,615.0,34.0,...,4.2,Other Snacks,Thai,4.553877,5.010635,5.087596,5.493061,5.072943,0.326254,0.000000
434135,137,2306,6,339.53,338.53,0,0,0.0,590.0,56.0,...,4.4,Pasta,Italian,0.000000,2.772589,3.761200,5.971262,4.562813,1.552592,0.295395
212986,66,2704,127,321.13,320.13,0,0,121.0,648.0,34.0,...,4.1,Other Snacks,Thai,4.804021,4.007333,5.087596,5.093750,4.609294,0.564507,0.312373
85996,32,1247,12,484.03,484.03,0,0,14.0,526.0,34.0,...,3.8,Biryani,Indian,2.708050,3.367296,3.367296,2.639057,3.003177,0.420449,0.000000
393747,110,2664,73,326.89,328.89,0,0,337.0,485.0,77.0,...,3.8,Salad,Italian,5.823046,6.003887,6.244167,6.008813,5.859663,0.465622,-0.608106
260427,76,2707,8,228.95,228.95,0,0,433.0,614.0,85.0,...,3.0,Beverages,Italian,6.073045,5.902633,5.866468,6.214608,6.087974,0.243565,0.000000
493586,161,2539,7,116.46,116.46,0,0,136.0,658.0,34.0,...,3.9,Beverages,Thai,4.919981,5.017280,4.812184,5.099866,5.042769,0.179510,0.000000
454441,145,2539,12,116.43,115.43,0,0,404.0,620.0,77.0,...,3.9,Beverages,Thai,6.003887,5.384495,5.823046,6.318968,5.725447,0.447459,0.866326


Fill NaNs in lag/rolling features with 0.
Rationale: a NaN here means "this center-meal pair doesn't have enough history yet" (new meal/center, or early weeks of the panel). We assume "no prior history" = "no prior demand observed" = 0. This is a modeling choice, not a fact — it deliberately keeps cold-start rows in training so the model learns to handle new center-meal pairs, rather than only ever seeing pairs with an established order history.

In [23]:
lag_roll_cols = ['lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'rolling_std_4']

# Add a flag so the model can distinguish "genuinely 0 prior demand" from "0 because we're filling missing history"
df_full['has_history_4wk'] = df_full['lag_4'].notna().astype(int)

for col in lag_roll_cols:
    df_full[col] = df_full[col].fillna(0)

df_full.head(10)

,center_id,meal_id,week,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,...,category,cuisine,log_num_orders,lag_1,lag_2,lag_4,rolling_mean_4,rolling_std_4,price_change_pct,has_history_4wk
0,10,1062,1,181.39,181.39,0,0,865.0,590.0,56.0,...,Beverages,Italian,6.763885,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
1,10,1062,2,183.36,182.36,0,0,782.0,590.0,56.0,...,Beverages,Italian,6.663133,6.763885,0.000000,0.000000,6.763885,0.000000,0.548366,0
2,10,1062,3,184.36,182.36,0,0,851.0,590.0,56.0,...,Beverages,Italian,6.747587,6.663133,6.763885,0.000000,6.713509,0.071243,1.096732,0
3,10,1062,4,182.36,183.36,0,0,1202.0,590.0,56.0,...,Beverages,Italian,7.092574,6.747587,6.663133,0.000000,6.724868,0.054082,-0.545375,0
4,10,1062,5,183.39,181.39,0,0,958.0,590.0,56.0,...,Beverages,Italian,6.865891,7.092574,6.747587,6.763885,6.816794,0.189081,1.102597,1
5,10,1062,6,162.05,183.39,0,0,1094.0,590.0,56.0,...,Beverages,Italian,6.998510,6.865891,7.092574,6.663133,6.842296,0.186427,-11.636403,1
6,10,1062,7,160.08,183.39,0,0,1513.0,590.0,56.0,...,Beverages,Italian,7.322510,6.998510,6.865891,6.747587,6.926140,0.151051,-12.710617,1
7,10,1062,8,160.05,182.39,0,0,1149.0,590.0,56.0,...,Beverages,Italian,7.047517,7.322510,6.998510,7.092574,7.069871,0.192391,-12.248479,1
8,10,1062,9,162.05,182.39,0,0,1282.0,590.0,56.0,...,Beverages,Italian,7.156956,7.047517,7.322510,6.865891,7.058607,0.191937,-11.151927,1
9,10,1062,10,161.05,181.39,0,0,1473.0,590.0,56.0,...,Beverages,Italian,7.295735,7.156956,7.047517,6.998510,7.131373,0.143611,-11.213408,1


In [24]:
# Captures annual seasonality (weeks 1 to 52)
df_full['week_of_year'] = ((df_full['week'] - 1) % 52) + 1

# Let the tree know if overall platform volume is growing over the 3 years
df_full['week_num'] = df_full['week']

In [25]:
df_full.to_csv('../data/processed/processed_data.csv', index=False)